# Image Processing with Pillow

In this notebook, we will process images using the Python Image Library (PIL); also known as Pillow.

In [ ]:
import sys
if 'google.colab' in sys.modules:
    %pip install numpy Pillow
else:
    !pip install numpy Pillow


In [2]:
import numpy as np  # numpy==1.26.4
from PIL import Image  # Pillow==10.2.0

## Pillow Concepts

In Pillow, there are few important concepts that we need to keep in mind. The first of which is that the convention of dimension orders is width first then height. Unlike NumPy arrays where height is the first axis then width.

In [4]:
img = Image.open('horse.png')

print(img.size)
print(np.asarray(img).shape)

(370, 316)
(316, 370, 3)


In [5]:
# img.save('./res/horse.png')  # converts the image format

> Refer to the [original source](https://pillow.readthedocs.io/en/stable/handbook/concepts.html) to learn more.

### Resampling Filters

For geometry operations that may map multiple input pixels to a single output pixel (image resize for example), the Python Imaging Library provides different resampling *filters* (also known as interpolation methods).

<img height="300" src="./res/interpolation.svg" alt="Source: https://en.wikipedia.org/wiki/Bicubic_interpolation">

> Note: All these filter are under `PIL.Image.Resampling`.

| Filter              | Downscaling quality | Upscaling quality | Performance |
|---------------------|---------------------|-------------------|-------------|
| `NEAREST`  |             |           | ⭐⭐⭐⭐⭐  |
| `BOX`      | ⭐          |           | ⭐⭐⭐⭐    |
| `BILINEAR` | ⭐          | ⭐        | ⭐⭐⭐      |
| `HAMMING`  | ⭐⭐        |           | ⭐⭐⭐      |
| `BICUBIC`  | ⭐⭐⭐      | ⭐⭐⭐    | ⭐⭐        |
| `LANCZOS`  | ⭐⭐⭐⭐    | ⭐⭐⭐⭐  | ⭐          |

The most common interpolation method used in deep learning is the `BILINEAR` interpolation.

In [ ]:
img.resize((437, 291), Image.Resampling.BOX).show()  # try other filters

### Image Modes

The ``mode`` of an image is a string which defines the type and depth of a pixel in the
image. Each pixel uses the full range of the bit depth. So a 1-bit pixel has a range of
0-1, an 8-bit pixel has a range of 0-255, a 32-signed integer pixel has the range of
INT32 and a 32-bit floating point pixel has the range of FLOAT32. The most relevant modes for us are:

- ``1`` (1-bit pixels, black and white, stored with one pixel per byte)
- ``L`` (8-bit pixels, grayscale)
- ``RGB`` (3x8-bit pixels, true color)
- ``RGBA`` (4x8-bit pixels, true color with transparency mask)
- ``I`` (32-bit signed integer pixels)
- ``F`` (32-bit floating point pixels)

In [6]:
print("Before:", img.getbands())

img = img.convert('RGBA')  # try other modes

print("After :", img.getbands())

img.show()

Before: ('R', 'G', 'B')
After : ('R', 'G', 'B', 'A')


### Transparency

In [9]:
img = img.convert("RGBA")

alpha = np.array(img.split()[-1])  # get the alpha channel
alpha[...] = 255  # reset the transparency to all white
alpha[240:310, 350:450] = 0  # select a crop to be transparent
alpha = Image.fromarray(alpha)  # convert into image
img.putalpha(alpha)  # add the alpha back to the image

print('Alpha Channel (transparency):')
alpha.show()

print('Image with transparency:')
img.show()

Alpha Channel (transparency):
Image with transparency:


## Pillow Modules

Pillow provides other modules to work with images. Here, we will explore some of them. The main module is [`Image`](https://pillow.readthedocs.io/en/stable/reference/Image.html) and it's [`Image` class](https://pillow.readthedocs.io/en/stable/reference/Image.html#the-image-class). You should explore its methods; such as `crop`, `resize`, `rotate`, and `paste`. Meanwhile, you can discover other modules like
[`ImageMath`](https://pillow.readthedocs.io/en/stable/reference/ImageMath.html),
[`ImageOps`](https://pillow.readthedocs.io/en/stable/reference/ImageOps.html),
[`ImageChops`](https://pillow.readthedocs.io/en/stable/reference/ImageChops.html), and
[`ImageMorph`](https://pillow.readthedocs.io/en/stable/reference/ImageOps.html)
in the [documentation](https://pillow.readthedocs.io/en/stable/reference/ImageMorph.html).

> Refer to this [tutorial](https://pillow.readthedocs.io/en/stable/handbook/tutorial.html) to learn more.

### ImageFilter

This module allows us to do convolution and filtering on images.

> Learn more [here](https://pillow.readthedocs.io/en/stable/reference/ImageFilter.html)

In [ ]:
from PIL import ImageFilter


img = img.convert("RGB")

#### Predefined Filters

##### BLUR

In [ ]:
img.filter(ImageFilter.BLUR)

##### SMOOTH

In [ ]:
img.filter(ImageFilter.SMOOTH)

In [ ]:
img.filter(ImageFilter.SMOOTH_MORE)

##### SHARPEN

In [ ]:
img.filter(ImageFilter.SHARPEN)

##### DETAIL

In [ ]:
img.filter(ImageFilter.DETAIL)

##### EDGE_ENHANCE

In [ ]:
img.filter(ImageFilter.EDGE_ENHANCE)

In [ ]:
img.filter(ImageFilter.EDGE_ENHANCE_MORE)

##### FINDE_EDGES

In [ ]:
img.filter(ImageFilter.FIND_EDGES)

##### CONTOUR

In [ ]:
img.filter(ImageFilter.CONTOUR)

##### EMBOSS

In [ ]:
img.filter(ImageFilter.EMBOSS)

#### Custom Filters

##### BoxBlur

In [ ]:
img.filter(ImageFilter.BoxBlur(radius=3))

##### GaussianBlur

In [ ]:
img.filter(ImageFilter.GaussianBlur(radius=3))

##### UnsharpenMask

In [ ]:
img.filter(ImageFilter.UnsharpMask(radius=2, percent=150, threshold=3))

##### Kernel (Convolution)

In [ ]:
kernel = np.array([
    [-1, -1, -1],
    [0, 1, 0],
    [1, 1, 1],
]) / 9
img.filter(ImageFilter.Kernel(kernel.shape[::-1], kernel.ravel()))

##### MinFilter

In [ ]:
img.filter(ImageFilter.MinFilter(size=3))

##### MedianFilter

In [ ]:
img.filter(ImageFilter.MedianFilter(size=3))

##### MaxFilter

In [ ]:
img.filter(ImageFilter.MaxFilter(size=3))

##### RankFilter

In [ ]:
img.filter(ImageFilter.RankFilter(size=3, rank=0))

##### ModeFilter

In [ ]:
img.filter(ImageFilter.ModeFilter(size=3))

### ImageEnhance

This module allows us to do simple image enhancements.

> Learn more [here](https://pillow.readthedocs.io/en/stable/reference/ImageEnhance.html)

In [ ]:
from PIL import ImageEnhance

#### Color

In [ ]:
ImageEnhance.Color(img).enhance(factor=3)

#### Contrast

In [ ]:
ImageEnhance.Contrast(img).enhance(factor=1.3)

#### Brightness

In [ ]:
ImageEnhance.Brightness(img).enhance(factor=1.5)

#### Sharpness

In [ ]:
ImageEnhance.Sharpness(img).enhance(factor=3)

### ImageColor

This module allows us to get colors using multiple formats.

> Learn more [here](https://pillow.readthedocs.io/en/stable/reference/ImageColor.html)

In [ ]:
from PIL import ImageColor


purple = ImageColor.getcolor('purple', mode='RGB')

It supports CSS3-style color specifiers:

- Common HTML color names (case insensitive)
- Red-Green-Blue (RGB)
- Hue-Saturation-Lightness (HSL)
- Hue-Saturation-Value (HSV)
- Hexadecimal values:

| binary | hex | decimal |
|--------|-----|---------|
| 0000 | 0 | 0  |
| 0001 | 1 | 1  |
| 0010 | 2 | 2  |
| 0011 | 3 | 3  |
| 0100 | 4 | 4  |
| 0101 | 5 | 5  |
| 0110 | 6 | 6  |
| 0111 | 7 | 7  |
| 1000 | 8 | 8  |
| 1001 | 9 | 9  |
| 1010 | a | 10 |
| 1011 | b | 11 |
| 1100 | c | 12 |
| 1101 | d | 13 |
| 1110 | e | 14 |
| 1111 | f | 15 |

For example, `'#fc'` = $(\text{f} \times 16) + \text{c} = (15 \times 16) + 12 = 252$.

In [ ]:
print('red  :', ImageColor.getrgb('red'))
print('green:', ImageColor.getrgb('rgb(0%, 100%, 0%)'))
print('white:', ImageColor.getrgb('hsl(360, 100%, 100%)'))
print('red  :', ImageColor.getrgb('hsv(360, 100%, 100%)'))
print('blue :', ImageColor.getrgb('rgb(0%, 0%, 100%)'))
print('green:', ImageColor.getrgb('#00ff00'))

### ImageDraw

This module allows us to draw on images.

> Learn more [here](https://pillow.readthedocs.io/en/stable/reference/ImageDraw.html)

In [ ]:
from PIL import ImageDraw


thickness = 3
color = ImageColor.getrgb("red")
with Image.open("./res/horse.jpg") as im:
    draw = ImageDraw.Draw(im)

    # left-diagonal line
    draw.line((0, 0) + im.size, width=thickness, fill=color)

    # right-diagonal line
    draw.line((0, im.height, im.width, 0), width=thickness, fill=color)

    # text
    draw.text((im.width // 2 - 10, im.height // 2 - 5), "Horse", fill=0)

    # circle
    top_left = (im.width // 2 - 30, im.height // 2 - 30)
    bottom_right = (im.width // 2 + 30, im.height // 2 + 30)
    draw.ellipse((top_left, bottom_right), outline=color, width=thickness)

im.show()

## Extras

This section, provide extra useful tips and tricks.

### The canonical way of openning images

In [ ]:
from PIL import Image

# context manager to get the file descriptor for reading the image
with Image.open("./res/horse.jpg").convert("RGB") as img:  # mode conversion is optional
    img.load()  # the image pixel values were not loaded until this line

img.show()  # the image can now be used and displayed outside the context

### Reading images from a link

In [ ]:
import requests  # pip install requests=2.31.0

url = 'https://pillow.readthedocs.io/en/stable/_static/pillow-logo.png'

img = Image.open(requests.get(url, stream=True).raw)

img.show()

### Draw text as an image

In [ ]:
from PIL import Image, ImageFont


def draw_text(text, size=None, font=None):
    # get the font
    if font is None:
        font = ImageFont.load_default(size=size)
    elif not isinstance(font, ImageFont.FreeTypeFont):
        font = ImageFont.truetype(font)

    # draw the text as a mask
    mask = font.getmask(text)

    # wrap the mask as an image
    image = Image.new(mask.mode, mask.size, color=None)
    image.im = mask
    return image


draw_text('T5 Bootcamp', size=30).show()

## Your Turn

You have reached the end of this notebook. You can experiment with your newly acquired skills to do:

1. Load the `coins.jpg` image, which contains a pile of gold coins
2. Make the coins image transparent (apply thresholding, process the mask to add as an alpha channel)
3. Load the `horse.jpg` image and paste the coins on the ground (try to make it look realistic)

<details>
<summary>Solution</summary>

```python
import numpy as np
from PIL import Image, ImageFilter, ImageEnhance, ImageOps

coins = Image.open('./res/coins.jpg')

# convert the image to grayscale, then threshold
threshold = 251
mask = Image.fromarray(np.asarray(coins.convert('L')) < threshold)

# remove salt-and-pepper noise using a median filter
mask = mask.filter(ImageFilter.MedianFilter(7))

# erode the boundaries of the mask using a min filter
mask = mask.filter(ImageFilter.MinFilter(3))

# use the mask to make the coins transparent
coins.putalpha(mask)

# resize the coins to fit inside a small rectangle
coins = ImageOps.contain(coins, (50, 50))

# change the brightness of the coins to match the background
coins = ImageEnhance.Brightness(coins).enhance(0.7)

# load the horse image
img = Image.open('./res/horse.jpg')

# paste the coins in a specific location (with transparency)
img.paste(coins, (650, 500), mask=coins)

img.show()
```

</details>

⚠️ Remember, there are plenty of ways to achieve the same results. Don't get discouraged if your solution didn't match ours. Enjoy the journey! Happy coding ;)